# AR tuning experiments — repeated paths and Jones-sampled models

This notebook implements **experiments 2 and 3** under **Tuning experiments** in the [AR thesis chapter](../../../Abschlussarbeit_2021/LaTeX/chapters/3_application_to_time_series.tex): experiment 2 uses a configurable number of independent AR(3) realizations with fixed coefficients. Experiment 3 draws fresh coefficient vectors using Jones’s method and simulates one path for each vector. Both experiments use one common fixed starting direction and tune absolute geometric steps.

For experiment 2, the fixed model is $X_t=0.6X_{t-1}-0.3X_{t-2}+0.2X_{t-3}+\varepsilon_t$, with $T=500$ and independent $\varepsilon_t\sim U(-1,1)$. The same coefficient vector appears in [Arbúes et al. (2013), §5.3, p. 13, with $b=0$](https://www.ucm.es/data/cont/docs/518-2013-11-05-Arbues_Feb19.pdf#page=13). It is a simulation precedent for the coefficients; the innovation choice here follows the thesis.

**Dimension convention:** newest-first windows have $k=p+1=4$ coordinates, so the starting direction is on $S^3$. The thesis currently writes $S^2$ in this paragraph; this notebook uses the four-dimensional embedding and does not edit the thesis.

The optimizer and grid search live in shared modules: [descent.py](../ar_tools/descent.py) and [tuning.py](../ar_tools/tuning.py). Each displayed figure is also exported as PDF and PNG. Experiments 2 and 3 select all configurations with the highest absolute-gap co-winner count. Experiment 2 additionally reports a separate coefficient-error co-winner count using the known simulation coefficients; the width counter is unchanged.

A final win-frequency comparison awards +1 to every configuration whose achieved minimum empirical width is within an absolute tolerance $\varepsilon=10^{-8}$ of the best achieved value on each realization. Win-count rankings and matrices are shown for the repeated-path experiments. Experiment 2 adds coefficient MAE counts with their own tolerance $\varepsilon_{\mathrm{MAE}}=10^{-2}$.
Both experiments use a **2,000-iteration cap and patience 200**.


In [ ]:
import hashlib
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import display

AR_CODE_DIR = next(
    candidate.resolve()
    for root in (Path.cwd(), *Path.cwd().parents)
    for candidate in (root / "Code" / "Code_AR_processes", root)
    if (candidate / "ar_tools/functions.py").is_file()
)
PROJECT_ROOT = AR_CODE_DIR.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Refresh the optimizer as well as the tuner in an existing notebook kernel.
from Code.Code_AR_processes.ar_tools import descent as descent_tools
descent_tools = importlib.reload(descent_tools)
from Code.Code_AR_processes.ar_tools.functions import is_stationary
# Refresh the local tuning helpers when setup is rerun in an existing kernel.
from Code.Code_AR_processes.ar_tools import tuning as tuning_tools
tuning_tools = importlib.reload(tuning_tools)
from Code.Code_AR_processes.ar_tools.tuning import (
    tune_fixed_ar_realizations, tune_jones_ar_realizations,
    summarize_configuration_wins,
)
with plt.rc_context():
    from Code.Code_AR_processes.ar_tools import tuning_plots as tuning_plot_tools
    tuning_plot_tools = importlib.reload(tuning_plot_tools)
    from Code.Code_AR_processes.ar_tools.tuning_plots import plot_tuning_matrices

if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"font.size": 10, "pdf.fonttype": 42})
np.set_printoptions(precision=8, suppress=True)
EXPERIMENT_NAME = "ar_tuning_experiments"
OUTPUT_ROOT = AR_CODE_DIR / "outputs"

from Code.Code_AR_processes.ar_tools.coefficient_errors import coefficient_mae


## Fixed settings and tuning grids

The grid is exactly $H\in\{2,5,10,20,50,100,200,1000\}$ and $s\in\{0.1,0.2,\ldots,0.9\}$, with $\beta=2^{-1/H}$. Each grid value $s$ is used directly as the initial step $\gamma_0$.

Both experiments use a **2,000-iteration cap and patience 200**. `improvement_tolerance=0` records every strictly smaller width and resets the patience counter. The win-frequency comparison awards +1 to every configuration whose achieved width is at most the best achieved width plus `epsilon_win_tolerance` (default $10^{-8}$). The simulator discards 2,000 burn-in observations, a retained convention from the previous tuning notebook. Seeds and burn-in are explicit experimental settings, not fitted parameters. [Thesis: termination and tuning paragraphs](../../../Abschlussarbeit_2021/LaTeX/chapters/3_application_to_time_series.tex), [simulation](../ar_tools/functions.py), [stopping rule](../ar_tools/descent.py).

Set **`number_of_realizations`** in the next cell to choose the number of independent paths in experiment 2 (default: **100**). The simulation, win-count summaries and matrices all use this setting.
Experiment 2 also awards a separate +1 for coefficient MAE within `coefficient_epsilon_win_tolerance` (default $10^{-2}$) of that path's smallest coefficient error. These two tolerances are independently adjustable. [Counting implementation](../ar_tools/tuning.py), `summarize_configuration_wins`.


In [ ]:
T = 500
p = 3
phi = np.array([0.6, -0.3, 0.2], dtype=float)
burnin = 2_000
optimizer_seed = 20261019
embedding_dimension = p + 1

max_iterations = 2_000
patience = 200
improvement_tolerance = 0.0
scale_grid = np.arange(1, 10) / 10.0  # 0.1, 0.2, ..., 0.9
half_life_grid = np.array([2, 5, 10, 20, 50, 100, 200, 1000])

number_of_realizations = 100  # CHANGE THIS: number of independent paths in experiment 2.
realization_process_seed = 20261020

epsilon_win_tolerance = 1e-8  # CHANGE THIS: absolute distance from each path's best achieved width.
coefficient_epsilon_win_tolerance = 1e-2  # CHANGE THIS: distance from each path's smallest coefficient MAE.

if p != 3 or phi.shape != (p,) or embedding_dimension != 4:
    raise ValueError("This experiment requires an AR(3) model and four-coordinate windows.")
stationary, companion_eigenvalues = is_stationary(phi)
if not stationary or phi[-1] == 0:
    raise ValueError("Choose a causal AR(3) coefficient vector with nonzero phi_3.")
if T < embedding_dimension + 1 or burnin < 0:
    raise ValueError("Use enough observations to form at least two windows and nonnegative burn-in.")

if (isinstance(number_of_realizations, (bool, np.bool_))
        or not isinstance(number_of_realizations, (int, np.integer))
        or number_of_realizations < 1):
    raise ValueError("number_of_realizations must be a positive integer.")
for tolerance in (epsilon_win_tolerance, coefficient_epsilon_win_tolerance):
    if (isinstance(tolerance, (bool, np.bool_))
            or not isinstance(tolerance, (int, float, np.integer, np.floating))
            or not np.isfinite(tolerance) or tolerance < 0):
        raise ValueError("Both win tolerances must be finite nonnegative numbers.")


## Fix the common starting direction and export settings

Generate one standard Gaussian four-vector using `optimizer_seed` and normalize it to obtain `initial_direction`. Experiments 2 and 3 reuse this exact direction for every configuration and realization. This is the initialization construction used by the [shared descent implementation](../ar_tools/descent.py).

Each experiment simulates its own paths through the [shared runners](../ar_tools/tuning.py), `tune_fixed_ar_realizations` and `tune_jones_ar_realizations`. The setup below prepares the fixed-coefficient experiment's reproducible settings and PDF/PNG output directories.


In [ ]:
initial_direction = np.random.default_rng(optimizer_seed).normal(size=embedding_dimension)
initial_direction /= np.linalg.norm(initial_direction)

configuration = {
    "experiment": "fixed_phi_realizations", "coefficient_error_metric": "mae_per_coefficient_v1", "T": int(T), "p": int(p), "phi": phi.tolist(),
    "burnin": int(burnin), "innovation_distribution": "Uniform(-1,1)",
    "process_seed": int(realization_process_seed), "optimizer_seed": int(optimizer_seed),
    "initial_direction": initial_direction.tolist(), "starts": 1,
    "max_iterations": int(max_iterations), "stopping": "iteration_limit_or_patience", "patience": int(patience),
    "improvement_tolerance": float(improvement_tolerance),
    "scale_grid": scale_grid.tolist(), "half_life_grid": half_life_grid.tolist(),
    "step_rules": ["absolute"],
    "selection": "all highest absolute-gap co-winner counts; parameter labels order ties for display",
    "replications": int(number_of_realizations),
    "win_frequency": {"score": "minimum_width", "credit": "one_per_absolute_gap_qualifier",
                      "epsilon": float(epsilon_win_tolerance)},
    "coefficient_win_frequency": {"score": "coefficient_mae",
                                  "credit": "one_per_absolute_gap_qualifier",
                                  "epsilon": float(coefficient_epsilon_win_tolerance)},
    "source_sha256": hashlib.sha256(b"".join(
        (AR_CODE_DIR / "ar_tools" / name).read_bytes()
        for name in ["functions.py", "innovations.py", "descent.py", "tuning.py", "tuning_plots.py", "coefficient_errors.py"]
    )).hexdigest(),
}
run_id = hashlib.sha256(repr(sorted(configuration.items())).encode()).hexdigest()[:16]
PDF_DIR = OUTPUT_ROOT / "pdfs" / EXPERIMENT_NAME / run_id
IMAGE_DIR = OUTPUT_ROOT / "images" / EXPERIMENT_NAME / run_id
for directory in (PDF_DIR, IMAGE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print(f"Experiment 2 run {run_id}: {number_of_realizations} paths, T={T}, one fixed starting direction")
print(f"True coefficients: {phi}; companion spectral radius: {np.max(np.abs(companion_eigenvalues)):.8f}")
print(f"Fixed starting direction: {initial_direction}")

## Experiment 2: fresh realizations, fixed coefficients and start

Keep $T=500$, $\phi=(0.6,-0.3,0.2)$, the innovation law, grids, budgets and **the fixed starting direction** from the shared setup. Only the innovation path changes. `number_of_realizations` controls the number of paths from a separate seed stream. Every configuration uses the same cloud within each replication. Each configuration uses the same initial step $\gamma_0$ on every path.

Record every configuration's smallest achieved empirical width on every path, then select **all configurations with the highest absolute-gap co-winner count** in the section below. The paired runs and per-path grid winners remain in `realization_results` and `realization_cases`. A separate coefficient-error counter compares the recovered coefficients with the known simulation coefficients, using its own tolerance. The two counters are reported separately. [Shared runner and counting rule](../ar_tools/tuning.py), `tune_fixed_ar_realizations(..., summarize=False)` and `summarize_configuration_wins`.

All runs use a **2,000-iteration cap and patience 200**. Selection uses no external optimum calculation or relative-gap target. Results stay in the notebook and memory; only PDF/PNG figures are exported. This experiment varies innovations conditional on one model and starting direction. Experiment 3 varies the coefficients; random-start sensitivity remains future work.

Each progress line reports the coefficient estimates and their MAE for that path's lowest-width grid result. The table below the run shows the same diagnostics; the shared [runner](../ar_tools/tuning.py) computes them from the selected direction.


In [ ]:
realization_experiment = tune_fixed_ar_realizations(
    phi, T, initial_direction, scale_grid, half_life_grid,
    replications=number_of_realizations, process_seed=realization_process_seed,
    burnin=burnin, iterations=max_iterations, patience=patience,
    improvement_tolerance=improvement_tolerance, optimizer_seed=optimizer_seed,
    progress=True, summarize=False,
)
realization_results = pd.DataFrame(realization_experiment["results"])
realization_cases = pd.DataFrame(realization_experiment["cases"])
np.testing.assert_allclose(realization_experiment["initial_direction"], initial_direction, atol=1e-15, rtol=0)
assert len(realization_results) == number_of_realizations * len(scale_grid) * len(half_life_grid)
assert realization_results["total_evaluations"].between(1, max_iterations).all()
assert realization_cases["process_seed"].is_unique
print(f"Completed {len(realization_cases)} fresh paths and {len(realization_results)} labeled runs.")
print(f"Original stopping rule: at most {max_iterations} evaluated iterations, patience={patience}.")
print(f"Fixed starting direction: {initial_direction}")
print("Per-path attained-width grid winners (diagnostics):")
realization_case_columns = [
    "replication", "T", "process_seed", "grid_best_width",
    "estimated_phi_1", "estimated_phi_2", "estimated_phi_3", "coefficient_mae",
    "grid_winner_step_value", "grid_winner_half_life", "runtime_seconds",
]
with pd.option_context("display.max_columns", None, "display.precision", 8):
    display(realization_cases[realization_case_columns])


## Win frequency: absolute-gap co-winners

On each of the **same configured realizations** from experiment 2, record each configuration's smallest achieved empirical width:
$$f_r(\theta)=\min_{t\,\mathrm{evaluated}}\hat w_{r,T}(u_{r,t}(\theta)).$$
Award a full **+1 to every configuration** satisfying
$$f_r(\theta)\le\min_{\eta\in\Theta}f_r(\eta)+\varepsilon,\qquad \varepsilon=10^{-8},$$
and count these credits over the configured paths:
$$N_\varepsilon(\theta)=\sum_{r=1}^R\mathbf{1}\!\left\{f_r(\theta)\le\min_{\eta\in\Theta}f_r(\eta)+\varepsilon\right\}.$$
The reference is the best achieved value among the tested configurations on the same path. Scores are compared at full precision against this absolute threshold. This is the defined counting rule for this experiment. [Counting implementation](../ar_tools/tuning.py), `summarize_configuration_wins(..., epsilon=...)`.

Change **`epsilon_win_tolerance`** in the settings cell to choose the absolute tolerance. Reuse `realization_results` from the **2,000-iteration cap / patience 200** descents above, with the same grid, starting direction and process seeds. The optimizer retains its existing settings. All qualifying configurations receive a full point regardless of speed; zero-win configurations remain in the ranking. Each configuration can receive at most $R$ points, while the total across configurations can exceed $R$. All configurations with the highest final count are displayed and highlighted. Equal counts are ordered by parameter labels for display only; every tied leader is retained.

Tolerance-based comparison with a best achieved objective has precedent in [Moré and Wild, §2.1, equation (2.2)](https://optimization-online.org/wp-content/uploads/2008/01/1883.pdf#page=7); their tolerance is scaled by the objective reduction. The fixed absolute tolerance and full +1 credit used here are experimental choices. Tables stay in the notebook; only PDF/PNG figures are exported.

In [ ]:
epsilon_win_experiment = summarize_configuration_wins(
    realization_results.to_dict("records"),
    score_key="minimum_width", epsilon=epsilon_win_tolerance,
)
epsilon_win_step_ranges = realization_results.groupby("candidate_id", as_index=False).agg(
    gamma0_min=("gamma0", "min"), gamma0_max=("gamma0", "max"),
)
epsilon_win_ranking = pd.DataFrame(epsilon_win_experiment["ranking"]).rename(
    columns={"rank": "win_rank"},
).merge(epsilon_win_step_ranges, on="candidate_id", validate="one_to_one").sort_values("win_rank")
epsilon_win_leaders = epsilon_win_ranking.loc[
    epsilon_win_ranking["wins"] == epsilon_win_ranking["wins"].max()
].copy()
epsilon_win_winners = pd.DataFrame(epsilon_win_experiment["winners"])
epsilon_win_cases = pd.DataFrame(epsilon_win_experiment["case_summaries"])
assert epsilon_win_experiment["epsilon"] == epsilon_win_tolerance
assert epsilon_win_experiment["cases"] == number_of_realizations
assert epsilon_win_ranking["wins"].between(0, number_of_realizations).all()
assert epsilon_win_ranking["wins"].sum() == len(epsilon_win_winners) == epsilon_win_experiment["total_points"]
assert epsilon_win_cases["co_winners"].sum() == len(epsilon_win_winners)
assert not epsilon_win_winners.duplicated(["replication", "T", "candidate_id"]).any()
assert (epsilon_win_winners["winning_score"] <= epsilon_win_winners["winning_threshold"]).all()

epsilon_win_columns = [
    "win_rank", "step_rule", "step_value", "half_life", "beta",
    "wins", "win_rate", "mean_evaluations",
]

print(f"Tuning {number_of_realizations} paths: cap {max_iterations}, patience {patience}.")
print(f"+1 to EVERY configuration with achieved width <= path minimum + {epsilon_win_tolerance:g}.")
with pd.option_context("display.max_columns", None, "display.precision", 10):
    display(epsilon_win_ranking[epsilon_win_columns].head(15))
    print("All configurations with the highest absolute-gap win count:")
    display(epsilon_win_leaders[epsilon_win_columns])
print(f"{epsilon_win_experiment['cases']} realizations; {epsilon_win_experiment['total_points']} full +1 credits; "
      f"{epsilon_win_experiment['tied_cases']} paths with multiple co-winners; "
      f"maximum {epsilon_win_experiment['maximum_co_winners']} co-winners on one path.")
for choice in epsilon_win_leaders.to_dict("records"):
    print(f"Absolute-gap win-count leader: {choice['step_rule']}, scale={choice['step_value']:g}, "
          f"H={choice['half_life']:g}, beta={choice['beta']:.12g}; "
          f"wins={int(choice['wins'])}/{number_of_realizations}.")

### Coefficient MAE co-winners on the same paths

For this simulated experiment, define the coefficient error of each configuration's recorded minimum-width direction by
$$e_r(\theta)=\frac1p\|\hat\phi_r(\theta)-\phi\|_1
=\frac1p\sum_{j=1}^{p}|\hat\phi_{r,j}(\theta)-\phi_j|,\qquad
\hat\phi_{r,j}(\theta)=-\frac{u_{r,j}(\theta)}{u_{r,0}(\theta)}.$$
This is the experiment's error diagnostic, computed from the returned directions in [the shared grid implementation](../ar_tools/tuning.py), `tune_geometric_half_life_schedules`.

Independently of the width counter, award **+1 to every configuration** satisfying
$$e_r(\theta)\le\min_{\eta\in\Theta:\ e_r(\eta)\ \mathrm{finite}}e_r(\eta)
+\varepsilon_{\mathrm{MAE}},\qquad\varepsilon_{\mathrm{MAE}}=10^{-2},$$
then add the credits across all configured paths. Change **`coefficient_epsilon_win_tolerance`** in the settings cell to adjust this tolerance separately from `epsilon_win_tolerance` ($10^{-8}$ for widths). The minimum is taken across the absolute-step grid on each path, and every qualifying configuration gets a full point, including ties. All comparisons use full precision. This counting rule is an experimental choice; [implementation](../ar_tools/tuning.py), `summarize_configuration_wins(..., score_key="coefficient_mae", allow_nonfinite=True)`.

Reuse the same recorded minimum-width directions: no new simulations or descents are needed. If $|u_{r,0}|\le10^{-10}$ or coefficients are nonfinite, report the coefficient error as undefined and award no coefficient point; an entirely undefined path awards no coefficient points. Undefined counts remain visible. The underlying width results remain available. Coefficient-error selection uses the known simulation truth.

The matrix below displays **width-win counts (W)** above **coefficient-error win counts (MAE)** in each cell. Red boxes mark all highest width counts; green boxes mark all highest coefficient-error counts. Nested boxes show a shared leader. Cell colors encode width counts. [Matrix plotting implementation](../ar_tools/tuning_plots.py), `plot_tuning_matrices`.


In [ ]:
# Recover coefficients from each configuration's recorded minimum-width direction.
realization_coefficient_diagnostics = realization_results.copy()
realization_directions = realization_results[[f"u_{j}" for j in range(p + 1)]].to_numpy(dtype=float)
realization_coefficients = np.full((len(realization_results), p), np.nan)
realization_coefficient_valid = (
    np.isfinite(realization_directions).all(axis=1) & (np.abs(realization_directions[:, 0]) > 1e-10)
)
with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
    np.divide(-realization_directions[:, 1:], realization_directions[:, :1],
              out=realization_coefficients, where=realization_coefficient_valid[:, None])
    realization_coefficient_errors = coefficient_mae(realization_coefficients, phi)
realization_coefficient_valid &= (
    np.isfinite(realization_coefficients).all(axis=1) & np.isfinite(realization_coefficient_errors)
)
realization_coefficient_errors[~realization_coefficient_valid] = np.nan
for j in range(p):
    realization_coefficient_diagnostics[f"estimated_phi_{j+1}"] = realization_coefficients[:, j]
realization_coefficient_diagnostics["coefficient_valid"] = realization_coefficient_valid
realization_coefficient_diagnostics["coefficient_mae"] = realization_coefficient_errors

coefficient_win_experiment = summarize_configuration_wins(
    realization_coefficient_diagnostics.to_dict("records"), score_key="coefficient_mae",
    epsilon=coefficient_epsilon_win_tolerance, allow_nonfinite=True,
)
coefficient_win_ranking = pd.DataFrame(coefficient_win_experiment["ranking"]).rename(
    columns={"rank": "coefficient_win_rank", "wins": "coefficient_wins",
             "win_rate": "coefficient_win_rate", "undefined_scores": "undefined_coefficients"},
)
coefficient_win_leaders = coefficient_win_ranking.loc[
    coefficient_win_ranking["coefficient_wins"] == coefficient_win_ranking["coefficient_wins"].max()
].copy()
coefficient_win_winners = pd.DataFrame(coefficient_win_experiment["winners"])
coefficient_win_cases = pd.DataFrame(coefficient_win_experiment["case_summaries"])
combined_win_ranking = epsilon_win_ranking.merge(
    coefficient_win_ranking[["candidate_id", "coefficient_win_rank", "coefficient_wins",
                             "coefficient_win_rate", "undefined_coefficients"]],
    on="candidate_id", validate="one_to_one",
)
assert coefficient_win_experiment["epsilon"] == coefficient_epsilon_win_tolerance
assert coefficient_win_experiment["cases"] == number_of_realizations
assert coefficient_win_ranking["coefficient_wins"].between(0, number_of_realizations).all()
assert coefficient_win_ranking["coefficient_wins"].sum() == len(coefficient_win_winners)
assert coefficient_win_cases["co_winners"].sum() == coefficient_win_experiment["total_points"]
if not coefficient_win_winners.empty:
    assert not coefficient_win_winners.duplicated(["replication", "T", "candidate_id"]).any()
    assert (coefficient_win_winners["winning_score"] <= coefficient_win_winners["winning_threshold"]).all()

coefficient_win_columns = [
    "coefficient_win_rank", "step_rule", "step_value", "half_life", "beta",
    "coefficient_wins", "coefficient_win_rate", "undefined_coefficients",
]
print(f"+1 to EVERY configuration with coefficient MAE <= path minimum + {coefficient_epsilon_win_tolerance:g}.")
with pd.option_context("display.max_columns", None, "display.precision", 10):
    display(coefficient_win_ranking[coefficient_win_columns].head(15))
    print("All configurations with the highest coefficient-error win count:")
    display(coefficient_win_leaders[coefficient_win_columns])
print(f"{coefficient_win_experiment['cases']} realizations; {coefficient_win_experiment['total_points']} full +1 coefficient credits; "
      f"{coefficient_win_experiment['tied_cases']} paths with multiple coefficient co-winners.")
print(f"Undefined coefficient estimates: {(~realization_coefficient_valid).sum()}; "
      f"paths with no finite coefficient estimate: {coefficient_win_experiment['undefined_cases']}.")


In [ ]:
epsilon_win_figure_stem = (
    f"07_epsilon_win_counts_width_{epsilon_win_tolerance:g}_coefficient_mae_{coefficient_epsilon_win_tolerance:g}"
)
fig, epsilon_win_matrices = plot_tuning_matrices(
    combined_win_ranking, scale_grid, half_life_grid, metric="wins", maximize=True, precision=0,
    secondary_metric="coefficient_wins",
    title=(f"Co-winner counts over {number_of_realizations} independent paths\n"
           f"Width epsilon={epsilon_win_tolerance:g}; coefficient MAE epsilon={coefficient_epsilon_win_tolerance:g}\n"
           f"+1 per qualifier on each path; cap {max_iterations}, patience {patience}"),
    colorbar_label="Width co-winner count W (higher is better)",
    filename=epsilon_win_figure_stem, pdf_directory=PDF_DIR, image_directory=IMAGE_DIR,
)
coefficient_win_matrices = {
    rule: combined_win_ranking.loc[combined_win_ranking["step_rule"] == rule].pivot(
        index="step_value", columns="half_life", values="coefficient_wins",
    ).reindex(index=scale_grid, columns=half_life_grid)
    for rule in ("absolute",)
}
plt.show()
plt.close(fig)


## Experiment 3: 100 Jones-sampled AR(3) models, one fixed starting direction

For each model, draw a fresh coefficient vector with the existing [Jones sampler](../ar_tools/functions.py), then simulate one independent length-$T=500$ path with $U(-1,1)$ innovations after the same 2,000-observation burn-in. Jones's construction samples coefficients uniformly from the causal AR region through independent beta-distributed partial autocorrelations and a recursive transformation. [Jones (1987), pp. 134–138](https://doi.org/10.2307/2347544).

Use **the common `initial_direction` from the shared setup**, one start per descent, the same scale and half-life grids, **2,000 maximum iterations and patience 200**. Each model's one fixed point cloud is shared by all absolute-step candidates. Coefficient and innovation draws use separate seeded streams. Record each draw's coefficients, partial autocorrelations, companion spectral radius and seeds. The Jones draws are retained without a spectral-radius cutoff.

Select configurations by the **same absolute-gap co-winner rule**: award +1 to every achieved minimum width at most that model's best achieved width plus `epsilon_win_tolerance` (default $10^{-8}$). [Shared tuning and counting implementation](../ar_tools/tuning.py), `tune_jones_ar_realizations` and `summarize_configuration_wins`.

The variables below use the `jones_` prefix. Earlier experiment results remain available. Only PDF/PNG matrices are exported; all sampled models and numerical results remain in the notebook. No iteration plots or JSON/CSV data exports are generated.

Each progress line also reports the theoretical **population** minimum $2/\sqrt{1+\|\phi\|^2}$ and the signed empirical-minus-population gap. The final width diagnostic compares this reference with a known feasible direction on the same sample. These benchmarks do not enter the tuning score. [Thesis, Proposition `prop:minimum_width`](../../../Abschlussarbeit_2021/LaTeX/chapters/3_application_to_time_series.tex).

In [ ]:
number_of_coefficient_realizations = 100  # CHANGE THIS: number of distinct Jones-sampled AR(3) models.
jones_coefficient_seed = 20261023
jones_process_seed = 20261024

jones_configuration = {
    "experiment": "jones_ar3_tuning", "T": int(T), "p": int(p),
    "replications": int(number_of_coefficient_realizations),
    "coefficient_sampling": "Jones1987_uniform_causal_AR3",
    "coefficient_seed": int(jones_coefficient_seed), "process_seed": int(jones_process_seed),
    "innovation_distribution": "Uniform(-1,1)", "burnin": int(burnin),
    "initial_direction": initial_direction.tolist(), "starts": 1, "optimizer_seed": int(optimizer_seed),
    "max_iterations": int(max_iterations), "patience": int(patience),
    "improvement_tolerance": float(improvement_tolerance),
    "scale_grid": scale_grid.tolist(), "half_life_grid": half_life_grid.tolist(),
    "step_rules": ["absolute"],
    "epsilon": float(epsilon_win_tolerance), "selection": "most_absolute_gap_co_winner_credits",
    "source_sha256": hashlib.sha256(b"".join(
        (AR_CODE_DIR / "ar_tools" / name).read_bytes()
        for name in ["functions.py", "innovations.py", "descent.py", "tuning.py", "tuning_plots.py", "coefficient_errors.py"]
    )).hexdigest(),
}
jones_run_id = hashlib.sha256(repr(sorted(jones_configuration.items())).encode()).hexdigest()[:16]
JONES_PDF_DIR = OUTPUT_ROOT / "pdfs" / EXPERIMENT_NAME / f"jones_{jones_run_id}"
JONES_IMAGE_DIR = OUTPUT_ROOT / "images" / EXPERIMENT_NAME / f"jones_{jones_run_id}"
for directory in (JONES_PDF_DIR, JONES_IMAGE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print(f"Experiment 3: {number_of_coefficient_realizations} Jones AR(3) models, T={T}; cap {max_iterations}, patience {patience}.")
print(f"The same fixed u_0 for every candidate and model: {initial_direction}")

In [ ]:
jones_experiment = tune_jones_ar_realizations(
    T, initial_direction, scale_grid, half_life_grid,
    replications=number_of_coefficient_realizations,
    coefficient_seed=jones_coefficient_seed, process_seed=jones_process_seed,
    burnin=burnin, iterations=max_iterations, patience=patience,
    improvement_tolerance=improvement_tolerance, optimizer_seed=optimizer_seed, progress=True, summarize=False,
)
jones_results = pd.DataFrame(jones_experiment["results"])
jones_cases = pd.DataFrame(jones_experiment["cases"])
np.testing.assert_allclose(jones_experiment["initial_direction"], initial_direction, atol=1e-15, rtol=0)
assert len(jones_cases) == number_of_coefficient_realizations
assert len(jones_results) == number_of_coefficient_realizations * len(scale_grid) * len(half_life_grid)
assert jones_results["total_evaluations"].between(1, max_iterations).all()
assert jones_results["selected_start"].eq(1).all()
assert jones_cases["coefficient_seed"].is_unique and jones_cases["process_seed"].is_unique
assert jones_cases[["phi_1", "phi_2", "phi_3"]].drop_duplicates().shape[0] == number_of_coefficient_realizations
assert jones_cases["spectral_radius"].between(0, 1, inclusive="left").all()
print(f"Completed {len(jones_cases)} distinct models and {len(jones_results)} paired candidate runs.")


In [ ]:
jones_model_columns = [
    "replication", "phi_1", "phi_2", "phi_3", "pacf_1", "pacf_2", "pacf_3",
    "spectral_radius", "coefficient_seed", "process_seed", "grid_best_width",
    "population_minimum_width", "grid_population_width_gap",
    "empirical_true_direction_width", "grid_empirical_true_direction_gap",
]
print("All sampled models and seeds:")
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.precision", 8):
    display(jones_cases[jones_model_columns])
print(f"Companion spectral-radius range: [{jones_cases['spectral_radius'].min():.8f}, "
      f"{jones_cases['spectral_radius'].max():.8f}].")

In [ ]:
jones_win_experiment = summarize_configuration_wins(
    jones_results.to_dict("records"), score_key="minimum_width", epsilon=epsilon_win_tolerance,
)
jones_win_step_ranges = jones_results.groupby("candidate_id", as_index=False).agg(
    gamma0_min=("gamma0", "min"), gamma0_max=("gamma0", "max"),
)
jones_win_ranking = pd.DataFrame(jones_win_experiment["ranking"]).rename(
    columns={"rank": "win_rank"},
).merge(jones_win_step_ranges, on="candidate_id", validate="one_to_one").sort_values("win_rank")
jones_win_leaders = jones_win_ranking.loc[
    jones_win_ranking["wins"] == jones_win_ranking["wins"].max()
].copy()
jones_win_winners = pd.DataFrame(jones_win_experiment["winners"])
assert jones_win_experiment["cases"] == number_of_coefficient_realizations
assert jones_win_ranking["wins"].between(0, number_of_coefficient_realizations).all()
assert jones_win_ranking["wins"].sum() == len(jones_win_winners) == jones_win_experiment["total_points"]
assert not jones_win_winners.duplicated(["replication", "T", "candidate_id"]).any()
assert (jones_win_winners["winning_score"] <= jones_win_winners["winning_threshold"]).all()
jones_win_columns = [
    "win_rank", "step_rule", "step_value", "half_life", "beta",
    "wins", "win_rate", "mean_evaluations",
]
print(f"Experiment 3 selection: +1 to every configuration within {epsilon_win_tolerance:g} of each model's best achieved width.")
with pd.option_context("display.max_columns", None, "display.precision", 10):
    display(jones_win_ranking[jones_win_columns].head(15))
    print("All configurations with the highest win count:")
    display(jones_win_leaders[jones_win_columns])
print(f"{jones_win_experiment['cases']} models; {jones_win_experiment['total_points']} full +1 credits; "
      f"{jones_win_experiment['tied_cases']} models with multiple co-winners.")

In [ ]:
jones_win_figure_stem = f"02_jones_win_counts_eps_{epsilon_win_tolerance:g}"
fig, jones_win_matrices = plot_tuning_matrices(
    jones_win_ranking, scale_grid, half_life_grid, metric="wins", maximize=True, precision=0,
    title=(f"Jones AR(3), {number_of_coefficient_realizations} models: +1 within {epsilon_win_tolerance:g}\n"
           f"Absolute steps; cap {max_iterations}, patience {patience}"),
    colorbar_label="Absolute-gap co-winner count (higher is better)",
    filename=jones_win_figure_stem, pdf_directory=JONES_PDF_DIR, image_directory=JONES_IMAGE_DIR,
)
plt.show()
plt.close(fig)

In [ ]:
print("Experiment 3 winners under the absolute-gap win-count rule:")
for choice in jones_win_leaders.to_dict("records"):
    print(f"{choice['step_rule']}: scale={choice['step_value']:g}, H={choice['half_life']:g}, "
          f"beta={choice['beta']:.12g}; {int(choice['wins'])}/{number_of_coefficient_realizations} model wins.")
    print(f"Actual gamma0 range across models: [{choice['gamma0_min']:.12g}, {choice['gamma0_max']:.12g}].")
print(f"One fixed u_0 throughout: {initial_direction}")
print(f"PDF matrices: {JONES_PDF_DIR}")

## Experiment 3: coefficient-recovery failures

Recover $\widehat\phi=-\widehat u_{1:}/\widehat u_0$ from each run's returned best-width direction. Apply the same diagnostic as the [stationarity-triggered restart experiment](../ar3_conditional_restarts_2000_200.ipynb): a finite estimate fails the stationarity check when its companion spectral radius is at least one. Undefined or nonfinite estimates are counted separately. [Coefficient transformation and stationarity check](../ar_tools/functions.py).

Also report per-vector coefficient MAE $\frac13\sum_{j=1}^3|\widehat\phi_j-\phi_j|>1$, the common coefficient-MAE diagnostic. This is a separate error threshold, not the stationarity test. Both are post-fit diagnostics: widths, parameter rankings, starts and stopping settings remain unchanged. The small-denominator mechanism was examined in the [earlier failure-tracking notebook](../ar3_failed_coefficient_tracking.ipynb); a failed stationarity check alone does not establish the mechanism of every error.

Report all grid runs, each win-count leader, and the single lowest-width grid result on each of the 100 model paths. These are different denominators. Tables stay in the notebook; no files or plots are exported.

In [ ]:
jones_coefficient_diagnostics = jones_results.copy()
jones_directions = jones_coefficient_diagnostics[[f"u_{j}" for j in range(embedding_dimension)]].to_numpy(dtype=float)
jones_true_coefficients = jones_coefficient_diagnostics[[f"phi_{j}" for j in range(1, p+1)]].to_numpy(dtype=float)
with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
    jones_recovered_coefficients = -jones_directions[:, 1:] / jones_directions[:, :1]
    jones_coefficient_mae = coefficient_mae(jones_recovered_coefficients, jones_true_coefficients)
jones_coefficients_finite = np.isfinite(jones_recovered_coefficients).all(axis=1)
jones_recovered_radii = np.full(len(jones_coefficient_diagnostics), np.nan)
for index in np.flatnonzero(jones_coefficients_finite):
    jones_recovered_radii[index] = np.max(np.abs(is_stationary(jones_recovered_coefficients[index])[1]))
for j in range(p):
    jones_coefficient_diagnostics[f"estimated_phi_{j+1}"] = jones_recovered_coefficients[:, j]
jones_coefficient_diagnostics["abs_first_direction_coordinate"] = np.abs(jones_directions[:, 0])
jones_coefficient_diagnostics["coefficient_mae"] = jones_coefficient_mae
jones_coefficient_diagnostics["estimated_spectral_radius"] = jones_recovered_radii
jones_coefficient_diagnostics["undefined_coefficients"] = ~jones_coefficients_finite
jones_coefficient_diagnostics["nonstationary_estimate"] = jones_coefficients_finite & (jones_recovered_radii >= 1)
jones_coefficient_diagnostics["failed_stationarity_check"] = (
    jones_coefficient_diagnostics["undefined_coefficients"] | jones_coefficient_diagnostics["nonstationary_estimate"]
)
jones_coefficient_diagnostics["coefficient_mae_above_1"] = jones_coefficients_finite & (jones_coefficient_mae > 1)

jones_failure_by_configuration = jones_coefficient_diagnostics.groupby(
    ["candidate_id", "step_rule", "step_value", "half_life", "beta"], as_index=False,
).agg(
    runs=("replication", "size"), nonstationary_estimates=("nonstationary_estimate", "sum"),
    undefined_coefficients=("undefined_coefficients", "sum"),
    failed_stationarity_checks=("failed_stationarity_check", "sum"),
    coefficient_mae_above_1=("coefficient_mae_above_1", "sum"),
    maximum_coefficient_mae=("coefficient_mae", "max"),
    minimum_abs_first_coordinate=("abs_first_direction_coordinate", "min"),
)
jones_failure_counts = pd.DataFrame([{
    "runs": len(jones_coefficient_diagnostics),
    "nonstationary_estimates": int(jones_coefficient_diagnostics["nonstationary_estimate"].sum()),
    "undefined_coefficients": int(jones_coefficient_diagnostics["undefined_coefficients"].sum()),
    "coefficient_mae_above_1": int(jones_coefficient_diagnostics["coefficient_mae_above_1"].sum()),
}])
jones_leading_failure_ids = jones_win_leaders["candidate_id"].tolist()
jones_leading_coefficient_failures = jones_failure_by_configuration.set_index("candidate_id").loc[
    jones_leading_failure_ids
].reset_index()
jones_grid_winner_coefficients = jones_cases[["replication", "T", "grid_winner_candidate_id"]].rename(
    columns={"grid_winner_candidate_id": "candidate_id"},
).merge(jones_coefficient_diagnostics, on=["replication", "T", "candidate_id"], validate="one_to_one")
assert len(jones_grid_winner_coefficients) == number_of_coefficient_realizations
assert jones_failure_by_configuration["runs"].eq(number_of_coefficient_realizations).all()
assert np.isfinite(jones_coefficient_diagnostics["minimum_width"]).all()
print("All grid runs (every tested configuration, including poor settings):")
with pd.option_context("display.max_columns", None):
    display(jones_failure_counts)
print("Win-count leaders:")
with pd.option_context("display.max_columns", None, "display.precision", 8):
    display(jones_leading_coefficient_failures)
print(f"Selecting the single lowest achieved width across the entire grid on each model: "
      f"{int(jones_grid_winner_coefficients['nonstationary_estimate'].sum())}/{number_of_coefficient_realizations} nonstationary estimates; "
      f"{int(jones_grid_winner_coefficients['undefined_coefficients'].sum())} undefined estimates; "
      f"{int(jones_grid_winner_coefficients['coefficient_mae_above_1'].sum())} with coefficient MAE > 1.")

## Experiment 3: achieved widths versus the population minimum

For each causal AR(3) model with innovations on $[-1,1]$, use the population reference from the [thesis, Proposition `prop:minimum_width`](../../../Abschlussarbeit_2021/LaTeX/chapters/3_application_to_time_series.tex):
$$m_{\mathrm{pop}}=\frac{2}{\sqrt{1+\|\phi\|^2}}.$$
The progress output and model table show the **signed discrepancy** $f_r(\theta)-m_{\mathrm{pop}}$. This compares an attained empirical width with a population minimum; it is not the optimization gap for the finite cloud and can be negative. The population reference is a diagnostic only.

For a comparison within the same empirical objective, compute the feasible true-coefficient direction
$$u_\phi=\frac{(1,-\phi_1,-\phi_2,-\phi_3)}{\sqrt{1+\|\phi\|^2}}$$
and its empirical width $\hat w_r(u_\phi)$. By the definition of an empirical minimum, $\hat m_r\le\hat w_r(u_\phi)$. Hence
$$f_r(\theta)-\hat m_r\;\ge\;f_r(\theta)-\hat w_r(u_\phi).$$
This inequality is a direct feasibility argument. A positive difference exceeding `epsilon_win_tolerance` demonstrates that the search returned a width worse than an available direction. The nonnegative part of this difference is a lower bound on empirical suboptimality. A smaller width does not establish that the empirical global minimum was reached.

The tables report both discrepancies for all 100 model paths and for the leading configurations. Counts of widths above either benchmark use the existing absolute tolerance $10^{-8}$; no extra numerical global optimization, coefficient-based filtering, or change in the tuning ranking is introduced. The phrase “failed completely” has no additional numerical cutoff here; the reported gaps show the size of each miss.

In [ ]:
jones_width_columns = [
    "replication", "grid_best_width", "population_minimum_width", "grid_population_width_gap",
    "empirical_true_direction_width", "grid_empirical_true_direction_gap",
]
jones_width_diagnostics = jones_cases[jones_width_columns].copy()
jones_width_diagnostics["grid_population_gap_percent"] = (
    100 * jones_width_diagnostics["grid_population_width_gap"] / jones_width_diagnostics["population_minimum_width"]
)
jones_width_diagnostics["empirical_suboptimality_lower_bound"] = (
    jones_width_diagnostics["grid_empirical_true_direction_gap"].clip(lower=0)
)
jones_width_diagnostics["above_population_benchmark"] = (
    jones_width_diagnostics["grid_population_width_gap"] > epsilon_win_tolerance
)
jones_width_diagnostics["worse_than_known_feasible_direction"] = (
    jones_width_diagnostics["grid_empirical_true_direction_gap"] > epsilon_win_tolerance
)
jones_width_summary = pd.DataFrame([{
    "models": len(jones_width_diagnostics),
    "grid_winners_above_population_minimum": int(jones_width_diagnostics["above_population_benchmark"].sum()),
    "grid_winners_worse_than_known_feasible_direction": int(jones_width_diagnostics["worse_than_known_feasible_direction"].sum()),
    "maximum_empirical_minus_population_gap": jones_width_diagnostics["grid_population_width_gap"].max(),
    "maximum_empirical_minus_population_percent": jones_width_diagnostics["grid_population_gap_percent"].max(),
    "maximum_empirical_suboptimality_lower_bound": jones_width_diagnostics["empirical_suboptimality_lower_bound"].max(),
}])
jones_run_width_diagnostics = jones_results.assign(
    above_population_benchmark=jones_results["population_width_gap"] > epsilon_win_tolerance,
    worse_than_known_feasible_direction=jones_results["empirical_true_direction_gap"] > epsilon_win_tolerance,
    population_gap_percent=100*jones_results["population_width_gap"]/jones_results["population_minimum_width"],
)
jones_width_by_configuration = jones_run_width_diagnostics.groupby(
    ["candidate_id", "step_rule", "step_value", "half_life"], as_index=False,
).agg(
    runs=("replication", "size"),
    above_population_benchmark=("above_population_benchmark", "sum"),
    worse_than_known_feasible_direction=("worse_than_known_feasible_direction", "sum"),
    maximum_population_gap_percent=("population_gap_percent", "max"),
)
jones_leading_width_diagnostics = jones_width_by_configuration.set_index("candidate_id").loc[
    jones_leading_failure_ids
].reset_index()
print("Best achieved width across the grid on each model: benchmark comparison.")
with pd.option_context("display.max_columns", None, "display.precision", 10):
    display(jones_width_summary)
    print("Win-count leaders:")
    display(jones_leading_width_diagnostics)
    print("Ten largest signed empirical-minus-population gaps among model-wise grid winners:")
    display(jones_width_diagnostics.sort_values("grid_population_width_gap", ascending=False).head(10))
print(f"{int(jones_width_diagnostics['above_population_benchmark'].sum())}/{number_of_coefficient_realizations} "
      f"model-wise grid winners are above the theoretical population minimum by more than {epsilon_win_tolerance:g}.")
print(f"{int(jones_width_diagnostics['worse_than_known_feasible_direction'].sum())}/{number_of_coefficient_realizations} "
      f"model-wise grid winners are worse than the known feasible true-coefficient direction by more than {epsilon_win_tolerance:g}.")